# Lecture 3 — Learning Willmore-Minimising Surface Maps

**XXII EGD School, Piauí, Brazil 2026**

Based on the paper [*Minimising Willmore Energy via Neural Flow*](https://arxiv.org/abs/2604.04321)  
E. Hirst · H. N. Sá Earp · T. S. R. Silva

---

### What you will build

Starting from nothing but PyTorch and NumPy, you will train a small neural network to
learn the surface embedding that **minimises the Willmore energy** over all embedded tori.
The theoretical minimum is

$$W = 2\pi^2 \approx 19.74,$$

achieved by the **Clifford torus**.
This inequality is the famous **Willmore conjecture**, posed by T. J. Willmore in 1965
and proved only in 2012 by F. C. Marques and A. Neves using min-max theory in differential
geometry ([*Ann. Math.* **179** (2014), 683–782](https://arxiv.org/abs/1202.6036)).

### Roadmap

| Step | Topic |
|------|-------|
| 1 | Imports & setup |
| 2 | Sampling the parameter domain |
| 3 | Fourier features for periodicity |
| 4 | Initial torus embedding in $\mathbb{R}^3$ |
| 5 | 3D visualisation |
| 6 | Supervised pretraining |
| 7 | Willmore energy loss |
| 8 | Regularity loss |
| 9 | PINN training |
| 10 | Loss curves & surface evolution |

No prior ML knowledge is assumed — every concept is introduced as it appears.

## 1 — Imports and Setup

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D   # noqa: F401 — registers 3-D projection

# Fix random seeds so results are reproducible
torch.manual_seed(42)
np.random.seed(42)

# Use CPU throughout (float32 is fine for this demo)
DEVICE = torch.device("cpu")
DTYPE  = torch.float32

PI = np.pi
TWO_PI = 2 * PI

print(f"PyTorch {torch.__version__}  |  device: {DEVICE}")

## 2 — Sampling the Fundamental Domain

A **torus** $T^2$ is topologically a square with opposite edges identified:

$$T^2 \cong [0, 2\pi] \times [0, 2\pi] \big/ \sim$$

where $(0, v) \sim (2\pi, v)$ and $(u, 0) \sim (u, 2\pi)$.

We work in this **parameter space** and train a neural network to learn the map
$$\varphi : (u, v) \mapsto (x, y, z) \in \mathbb{R}^3.$$

For training we need a cloud of points — random samples drawn uniformly
from $[0, 2\pi]^2$.  No mesh is required; this is the key advantage of PINNs over
classical finite-element methods.

In [ ]:
def sample_torus_domain(n: int) -> torch.Tensor:
    """
    Draw n points uniformly at random from [0, 2π] × [0, 2π].
    Returns a tensor of shape (n, 2).
    """
    return torch.rand(n, 2, dtype=DTYPE, device=DEVICE) * TWO_PI


# ── Visualise the domain ──────────────────────────────────────────────────────
N_SCATTER = 2000
uv = sample_torus_domain(N_SCATTER)

fig, ax = plt.subplots(figsize=(5, 5))

ax.scatter(uv[:, 0].numpy(), uv[:, 1].numpy(), s=2, alpha=0.4, color="steelblue")
ax.set_xlabel("$u$"); ax.set_ylabel("$v$")
ax.set_title("Fundamental Domain Sample  $[0, 2\\pi]^2$")
ax.set_aspect("equal")
# Draw identification boundaries
for x in [0, TWO_PI]:
    ax.axvline(x, color="crimson", lw=1.5, ls="--", label="identified" if x == 0 else "")
for y in [0, TWO_PI]:
    ax.axhline(y, color="navy", lw=1.5, ls="--")

plt.tight_layout()
plt.show()

print(f"Sample tensor shape: {uv.shape}   min: {uv.min():.3f}   max: {uv.max():.3f}")


### Exercise — Alternative Sampling Strategies

Write a new sampling function `sample_torus_domain_custom(n)` that replaces the
uniform random draw with a different strategy. Some ideas:

- **Uniform lattice** — instead of random points, place them on a regular grid.
  *Hint:* `torch.linspace` and `torch.meshgrid` may be useful.
- **Biased sample** — oversample near $u = 0$ or near the boundary, or concentrate
  points close to a curve of interest.
  *Hint:* apply a non-uniform transformation to uniform samples, e.g. squaring or
  using `torch.distributions`.
- **Quasi-random (low-discrepancy)** — look up *Halton sequences* or the
  *Sobol sequence* (`torch.quasirandom.SobolEngine`).

Visualise your samples the same way as above. How do different strategies affect
training? Does a biased sample help the network learn faster, or does it introduce
artefacts?


In [ ]:
### WRITE EXERCISE CODE HERE ###

## 3 — Fourier Feature Embeddings for Periodicity

### The problem

A standard multilayer perceptron (MLP) maps $\mathbb{R}^n \to \mathbb{R}^m$ and has no
built-in notion of periodicity.  If we feed raw $(u, v)$ as inputs, the network can
easily learn $\varphi(0, v) \neq \varphi(2\pi, v)$, violating the identification.

### The solution: Fourier features

Replace $(u, v)$ by its **Fourier feature vector**

$$\gamma(u, v) = \bigl(\sin u,\, \cos u,\, \sin v,\, \cos v,\;
                       \sin 2u,\, \cos 2u,\, \sin 2v,\, \cos 2v,\;
                       \ldots\bigr).$$

Because $\sin(k \cdot 0) = \sin(k \cdot 2\pi) = 0$ and similarly for cosine, the
feature vector is **identical** at identified boundary points.  Any downstream MLP
applied to $\gamma$ is therefore automatically periodic — no extra loss term required.

The number of frequencies $K$ controls expressiveness: $K = 1$ can only represent
simple shapes; larger $K$ is needed for fine surface detail.

In [ ]:
def fourier_features(uv: torch.Tensor, num_freqs: int = 4) -> torch.Tensor:
    """
    Map (u, v) → (sin u, cos u, sin v, cos v, sin 2u, cos 2u, …)
    for frequencies k = 1, …, num_freqs.

    Args:
        uv:        (N, 2) tensor with u = uv[:, 0],  v = uv[:, 1]
        num_freqs: number of frequency components K

    Returns:
        features:  (N, 4·K) tensor
    """
    parts = []
    for k in range(1, num_freqs + 1):
        parts.extend([
            torch.sin(k * uv[:, 0:1]),
            torch.cos(k * uv[:, 0:1]),
            torch.sin(k * uv[:, 1:2]),
            torch.cos(k * uv[:, 1:2]),
        ])
    return torch.cat(parts, dim=1)   # (N, 4·K)


# ── Feature dimension grows with K ────────────────────────────────────────────
for K in [1, 2, 4, 8]:
    f = fourier_features(uv[:4], num_freqs=K)
    print(f"  K={K}  →  feature dimension = {f.shape[1]}")

# ── Apply to the domain sample from Section 2 ────────────────────────────────
K_DEMO   = 4
features = fourier_features(uv, num_freqs=K_DEMO)

print(f"\nDomain sample  →  Fourier features (K={K_DEMO}):")
print(f"  Input  shape : {uv.shape}")
print(f"  Output shape : {features.shape}   (4 · {K_DEMO} = {4*K_DEMO} features per point)")
print(f"  Value range  : [{features.min().item():.4f}, {features.max().item():.4f}]")
print(f"  Mean abs val : {features.abs().mean().item():.4f}")
print(f"  Std dev      : {features.std().item():.4f}")


### Exercise — Numerical Periodicity Error

The Fourier features should satisfy $\gamma(0, v) = \gamma(2\pi, v)$ and $\gamma(u, 0) = \gamma(u, 2\pi)$ for all $u, v$ — but how close is "exact" on a computer using 32-bit floats?

**Task:** Draw $N = 500$ random values $v \in [0, 2\pi]$ and compute the **mean** absolute difference

$$\frac{1}{N} \sum_{i=1}^{N} \bigl\|\gamma(0, v_i) - \gamma(2\pi, v_i)\bigr\|_1.$$

What value do you expect? Is this consistent with float32 machine epsilon (~$10^{-7}$)?

*Hint:* Build two $(N, 2)$ tensors — one with $u = 0$, one with $u = 2\pi$ — using `torch.zeros` / `torch.full`, evaluate `fourier_features` on each, then call `.abs().mean()` on the difference.


In [ ]:
### WRITE EXERCISE CODE HERE ###

## 4 — Initial Embedding into $\mathbb{R}^3$

The **standard torus** with major radius $R$ and tube radius $r$ is

$$\varphi(u, v) = \bigl((R + r\cos v)\cos u,\;(R + r\cos v)\sin u,\; r\sin v\bigr).$$

The **Willmore energy** of a smooth surface $\Sigma \hookrightarrow \mathbb{R}^3$ is

$$W(\varphi) = \iint_\Sigma H^2 \, dA,$$

where $H$ is the mean curvature and $dA = \sqrt{EG - F^2}\,du\,dv$ is the area element
computed from the first fundamental form coefficients
$E = \langle\varphi_u,\varphi_u\rangle$,
$F = \langle\varphi_u,\varphi_v\rangle$,
$G = \langle\varphi_v,\varphi_v\rangle$.

For a torus of revolution the energy evaluates to

$$W(R, r) = \frac{\pi^2 R^2}{r\sqrt{R^2 - r^2}},$$

giving $W = 16\pi^2/\!\sqrt{15} \approx 40.8$ for $(R,r)=(4,1)$ and
$W = 2\pi^2 \approx 19.74$ for $R = \sqrt{2}$, $r = 1$ (the Clifford torus).

The sharp lower bound $W \geq 2\pi^2$ for any embedded torus — the **Willmore conjecture**,
posed in 1965 — was proved by Marques and Neves in 2012
([*Ann. Math.* **179**, 683–782](https://arxiv.org/abs/1202.6036)).

We use $(R, r) = (4, 1)$ as our **starting point** — the PINN will then drive $W$
down toward $2\pi^2$.


In [ ]:
# ── Analytic torus embedding ──────────────────────────────────────────────────
R_INIT, r_INIT = 4.0, 1.0   # starting torus (W = 16π²/√15 ≈ 40.8)
R_CLIFF        = np.sqrt(2)  # Clifford torus (W = 2π²)

def torus_embed(uv: torch.Tensor, R: float = R_INIT, r: float = r_INIT) -> torch.Tensor:
    """
    Standard torus: φ(u,v) = ((R + r cos v) cos u, (R + r cos v) sin u, r sin v)

    Args:
        uv: (N, 2)  parameter coordinates u = uv[:,0], v = uv[:,1]
    Returns:
        xyz: (N, 3)
    """
    u, v = uv[:, 0], uv[:, 1]
    x = (R + r * v.cos()) * u.cos()
    y = (R + r * v.cos()) * u.sin()
    z = r * v.sin()
    return torch.stack([x, y, z], dim=1)


# Evaluate on the domain sample from Section 2
xyz_init = torus_embed(uv)

print("Embedding shape :", xyz_init.shape)
print(f"x range : [{xyz_init[:,0].min():.2f}, {xyz_init[:,0].max():.2f}]")


## 5 — Visualising the Tori in 3D

We plot both the **starting torus** $(R=4, r=1)$ and the **target** Clifford torus
$(R=\sqrt{2}, r=1)$ side by side, colouring each surface by the analytic mean
curvature

$$H = -\frac{R + 2r\cos v}{2r(R + r\cos v)}.$$

This gives intuition about where curvature concentrates (the inner equator of the
tube) and how the two tori differ geometrically.


In [ ]:
def analytic_mean_curvature(V: np.ndarray, R: float, r: float) -> np.ndarray:
    """H = -(R + 2r cosv) / (2r(R + r cosv))  for a torus of revolution."""
    return -(R + 2 * r * np.cos(V)) / (2 * r * (R + r * np.cos(V)))


def plot_torus_3d(ax, R: float, r: float, title: str) -> None:
    """Scatter-plot the torus (R, r) using the domain sample `uv` from Section 2."""
    V_np = uv[:, 1].numpy()
    with torch.no_grad():
        xyz = torus_embed(uv, R=R, r=r).numpy()
    X, Y, Z = xyz[:, 0], xyz[:, 1], xyz[:, 2]

    H      = analytic_mean_curvature(V_np, R, r)
    H_norm = (H - H.min()) / (H.max() - H.min() + 1e-10)

    ax.scatter(X, Y, Z, c=H_norm, cmap="coolwarm", s=3, alpha=0.7)
    ax.set_title(title, pad=10)
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")

    # Equal axis scales — prevent the torus being visually squished
    half = max(X.max() - X.min(), Y.max() - Y.min(), Z.max() - Z.min()) / 2
    cx, cy, cz = (X.max()+X.min())/2, (Y.max()+Y.min())/2, (Z.max()+Z.min())/2
    ax.set_xlim(cx - half, cx + half)
    ax.set_ylim(cy - half, cy + half)
    ax.set_zlim(cz - half, cz + half)
    ax.set_box_aspect([1, 1, 1])

    sm = plt.cm.ScalarMappable(cmap="coolwarm",
                                norm=plt.Normalize(H.min(), H.max()))
    plt.colorbar(sm, ax=ax, shrink=0.5, pad=0.12, label="$H$ (mean curvature)")


fig = plt.figure(figsize=(12, 5))
ax1 = fig.add_subplot(121, projection="3d")
ax2 = fig.add_subplot(122, projection="3d")

plot_torus_3d(ax1, R=R_INIT,  r=r_INIT, title=f"Initial torus  $(R={R_INIT}, r={r_INIT})$")
plot_torus_3d(ax2, R=R_CLIFF, r=r_INIT, title=f"Clifford torus  $(R=\\sqrt{{2}},\\, r=1)$  — target")

plt.suptitle("Surface coloured by mean curvature $H$", y=1.02, fontsize=12)
plt.tight_layout()
plt.show()


### Exercise — Exploring Torus Geometry

**Try different radii** by changing `R_INIT` and `r_INIT` in section 4 and re-running from there.  Some starting points:

| $R$ | $r$ | Character |
|-----|-----|-----------|
| $\sqrt{2} \approx 1.41$ | $1$ | Clifford torus — Willmore minimum |
| $2$ | $1$ | "Standard" round torus |
| $4$ | $1$ | Current starting torus |
| $1.1$ | $1$ | Near-degenerate: inner circle almost pinches off |
| $4$ | $2$ | Fatter tube, same hole |

**Questions to think about:**
- How does the range of $H$ change as $R/r$ grows?  What happens geometrically as $R \to r^+$?
- Replace `uv` in `plot_torus_3d` with the custom sample you wrote in the section 2 exercise.  Does a lattice or a biased sample reveal more surface structure in the scatter plot than a uniform random draw?


## 6 — Supervised Pretraining

Before doing anything Willmore-related we **warm-start** the network to reproduce
the analytic embedding $\varphi_{\text{init}}(u,v)$ using plain supervised learning.

**Why?** If we started PINN training from a random network, the output would be a
chaotic cloud that is nowhere near a smooth surface.  Pretraining gives us a sensible
surface with the correct topology (the starting torus) as the initial condition for the physics training.

**Architecture:** `Fourier features → Linear(64) → Tanh → Linear(128) → Tanh → Linear(64) → Tanh → Linear(3)`.

**Loss:** mean squared error $\mathcal{L}_\text{sup} = \frac{1}{N}\sum_i \|\hat\varphi_i - \varphi_i\|^2$.

In [ ]:
# ── Network definition ────────────────────────────────────────────────────────

NUM_FREQS = 4    # Fourier frequencies per dimension

class TorusNet(nn.Module):
    """
    Periodic MLP:  (u,v) → Fourier features → tanh MLP → (x, y, z)

    The Fourier feature layer enforces φ(0,v) = φ(2π,v) and φ(u,0) = φ(u,2π)
    by construction, so no periodicity penalty is needed in the loss.
    """
    def __init__(self, num_freqs: int = NUM_FREQS,
                 hidden: list = [64, 128, 128, 64]):
        super().__init__()
        in_dim = 4 * num_freqs
        self.num_freqs = num_freqs

        layers = []
        prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.Tanh()]
            prev = h
        layers += [nn.Linear(prev, 3)]          # output: (x, y, z)
        self.net = nn.Sequential(*layers)

        # Xavier initialisation — good default for tanh networks
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, uv: torch.Tensor) -> torch.Tensor:
        return self.net(fourier_features(uv, self.num_freqs))


# ── Supervised training loop ──────────────────────────────────────────────────

def supervised_train(model: nn.Module,
                     R: float = R_INIT, r: float = r_INIT,
                     epochs: int = 300,
                     lr: float = 1e-3,
                     batch: int = 512) -> list:
    """Train model to match the analytic torus via MSE on random batches."""
    opt = optim.Adam(model.parameters(), lr=lr)
    losses = []
    for epoch in range(1, epochs + 1):
        uv  = sample_torus_domain(batch)
        xyz = torus_embed(uv, R, r)            # ground-truth targets

        pred = model(uv)
        loss = nn.functional.mse_loss(pred, xyz)

        opt.zero_grad()
        loss.backward()
        opt.step()

        losses.append(loss.item())
        if epoch % 60 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d}/{epochs}  supervised MSE = {loss.item():.5f}")
    return losses


torch.manual_seed(0)
model = TorusNet().to(DEVICE)

print(f"Network parameters: {sum(p.numel() for p in model.parameters()):,}")
print("── Supervised pretraining ──")
sup_losses = supervised_train(model, epochs=300)

# ── Verify the network matches the torus ─────────────────────────────────────
with torch.no_grad():
    uv_check = sample_torus_domain(2000)
    err = (model(uv_check) - torus_embed(uv_check)).norm(dim=1).mean().item()
print(f"\nMean reconstruction error (L2 per point): {err:.4f}")

# ── Visualise the supervised surface ─────────────────────────────────────────
uv_vis = sample_torus_domain(3000)
with torch.no_grad():
    xyz_net = model(uv_vis).numpy()

X, Y, Z = xyz_net[:, 0], xyz_net[:, 1], xyz_net[:, 2]
Z_norm  = (Z - Z.min()) / (Z.max() - Z.min() + 1e-10)

fig = plt.figure(figsize=(6, 5))
ax  = fig.add_subplot(111, projection="3d")
ax.scatter(X, Y, Z, c=Z_norm, cmap="coolwarm", s=3, alpha=0.7)
ax.set_title(f"Supervised network output  $(R={R_INIT}, r={r_INIT})$", pad=10)
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")

half = max(X.max() - X.min(), Y.max() - Y.min(), Z.max() - Z.min()) / 2
cx, cy, cz = (X.max()+X.min())/2, (Y.max()+Y.min())/2, (Z.max()+Z.min())/2
ax.set_xlim(cx - half, cx + half)
ax.set_ylim(cy - half, cy + half)
ax.set_zlim(cz - half, cz + half)
ax.set_box_aspect([1, 1, 1])

plt.tight_layout()
plt.show()

### Exercise — Pushing the Supervised Pretraining

Experiment with the `supervised_train` hyperparameters and observe the effect on the final reconstruction error and surface quality.

**Questions:**

1. **Loss floor** — How low can you get the supervised MSE?  Try increasing `epochs` to 1000 or 2000.  Does the loss keep falling or does it plateau?

2. **Batch size** — Change `batch` from 512 to 64, 256, 2048.  How does the training curve change?  Is a smaller batch noisier but faster to converge, or does it diverge?

3. **Learning rate** — Try `lr = 1e-2`, `1e-3` (default), `1e-4`.  What happens with a rate that is too large?  Too small?

4. **Network width** — The `hidden` list controls layer sizes.  Try a smaller network `[32, 64, 32]` and a larger one `[128, 256, 256, 128]`.  How does capacity affect the minimum achievable error?

*Hint:* Re-run from the `torch.manual_seed(0)` line each time to start from the same random initialisation, so comparisons are fair.  
*Note:* After supervised learning our result is a trained embedding map, as a neural network, we don't yet know how to get the mean curvature for this map so we colour by z-coordinate.


## 7 — The Willmore Energy Loss

### Differential geometry via autograd

We need $H$ and $\sqrt{EG-F^2}$ at each sampled point.  Rather than deriving closed-form
expressions, we let **PyTorch's automatic differentiation** compute them from the
network output $\varphi$. This is another benefit of using AI-techniques -- these computations are exceptionally efficient in these libraries.

The recipe is:

1. **First fundamental form** — evaluate $\varphi_u = \partial\varphi/\partial u$
   and $\varphi_v = \partial\varphi/\partial v$ with `torch.autograd.grad`.

   $$E = \langle\varphi_u,\varphi_u\rangle,\quad
     F = \langle\varphi_u,\varphi_v\rangle,\quad
     G = \langle\varphi_v,\varphi_v\rangle.$$

2. **Unit normal** — $\hat{n} = (\varphi_u \times \varphi_v)/|\varphi_u \times \varphi_v|$.

3. **Second fundamental form** — differentiate $\varphi_u$ and $\varphi_v$ once more, with a second `torch.autograd.grad`.

   $$L = \langle\varphi_{uu},\hat{n}\rangle,\quad
     M = \langle\varphi_{uv},\hat{n}\rangle,\quad
     N = \langle\varphi_{vv},\hat{n}\rangle.$$

4. **Mean curvature** —
   $H = \dfrac{EN - 2FM + GL}{2(EG - F^2)}.$

5. **Monte Carlo Willmore energy** — The Willmore energy is the integral of $H^2$ over the surface:
   $$W = \iint_{T^2} H^2\,dA = \iint_{[0,2\pi]^2} H^2\,\sqrt{EG-F^2}\;du\,dv.$$
   Sampling $(u_i,v_i) \stackrel{\mathrm{iid}}{\sim} \mathrm{Uniform}([0,2\pi]^2)$ and applying the Monte Carlo rule gives
   $$W \approx \frac{(2\pi)^2}{N}\sum_{i=1}^{N} H_i^2\,\sqrt{E_iG_i - F_i^2},$$
   i.e. the domain area $(2\pi)^2$ times the sample mean of $H^2\sqrt{EG-F^2}$.


In [ ]:
EPS = 1e-8   # numerical floor for denominators / norms


def _first_derivs(model: nn.Module, uv: torch.Tensor):
    """
    Compute φ(uv), φ_u, φ_v via autograd.

    We loop over the 3 output components and compute the gradient of each
    w.r.t. the 2-dimensional input.  This gives a (N,3) tensor for each
    partial derivative while keeping the computation graph intact for the
    second derivatives we need below.

    Returns: phi (N,3), phi_u (N,3), phi_v (N,3), uv_leaf (N,2)
    """
    uv_leaf = uv.detach().requires_grad_(True)
    phi = model(uv_leaf)                        # (N, 3)

    phi_u_cols, phi_v_cols = [], []
    for k in range(3):
        g = torch.autograd.grad(phi[:, k].sum(), uv_leaf,
                                create_graph=True, retain_graph=True)[0]  # (N,2)
        phi_u_cols.append(g[:, 0:1])
        phi_v_cols.append(g[:, 1:2])

    phi_u = torch.cat(phi_u_cols, dim=1)   # (N, 3)
    phi_v = torch.cat(phi_v_cols, dim=1)   # (N, 3)
    return phi, phi_u, phi_v, uv_leaf


def _second_derivs(phi_u: torch.Tensor, phi_v: torch.Tensor, uv_leaf: torch.Tensor):
    """
    Compute φ_uu, φ_uv, φ_vv by differentiating φ_u and φ_v once more.
    Returns: phi_uu, phi_uv, phi_vv  each of shape (N, 3)
    """
    phi_uu, phi_uv, phi_vv = [], [], []
    for k in range(3):
        g_u = torch.autograd.grad(phi_u[:, k].sum(), uv_leaf,
                                  create_graph=True, retain_graph=True)[0]
        phi_uu.append(g_u[:, 0:1])
        phi_uv.append(g_u[:, 1:2])

        g_v = torch.autograd.grad(phi_v[:, k].sum(), uv_leaf,
                                  create_graph=True, retain_graph=True)[0]
        phi_vv.append(g_v[:, 1:2])

    return (torch.cat(phi_uu, dim=1),
            torch.cat(phi_uv, dim=1),
            torch.cat(phi_vv, dim=1))


def compute_willmore(model: nn.Module, uv_batch: torch.Tensor) -> torch.Tensor:
    """
    Monte Carlo estimate  W ≈ (2π)² · mean(H² · √(EG−F²))
    over a batch of points drawn from [0, 2π]².
    Returns a differentiable scalar tensor.
    """
    _, phi_u, phi_v, uv_leaf = _first_derivs(model, uv_batch)

    # First fundamental form
    E_coef     = (phi_u * phi_u).sum(1)
    F_coef = (phi_u * phi_v).sum(1)
    G_coef     = (phi_v * phi_v).sum(1)
    det   = torch.clamp(E_coef * G_coef - F_coef ** 2, min=EPS)

    # Unit normal
    n_hat = torch.linalg.cross(phi_u, phi_v)
    n_hat = n_hat / n_hat.norm(dim=1, keepdim=True).clamp(min=EPS)

    # Second fundamental form
    phi_uu, phi_uv, phi_vv = _second_derivs(phi_u, phi_v, uv_leaf)
    L_coef  = (phi_uu * n_hat).sum(1)
    M_coef  = (phi_uv * n_hat).sum(1)
    N_coef = (phi_vv * n_hat).sum(1)

    # Mean curvature and area element
    H      = (E_coef * N_coef - 2 * F_coef * M_coef + G_coef * L_coef) / (2 * det)
    area_el = det.sqrt()

    return torch.mean(H ** 2 * area_el) * TWO_PI ** 2


# ── Evaluate on the pretrained (= initial) surface ───────────────────────────
uv_eval = sample_torus_domain(3000) #...use a new sample.
W_init  = compute_willmore(model, uv_eval)

print(f"Willmore energy on initial surface  W = {W_init.item():.4f}")
print(f"Analytic value for (R=4, r=1)       W = 16π²/√15 ≈ {16*PI**2/np.sqrt(15):.4f}")
print(f"Target (Clifford torus)             W = 2π²    ≈ {2*PI**2:.4f}")
print("...note the supervised Willmore is much higher than the target, but also than the analytic,")
print("this is because MSE loss only trains to match position and not derivatives.")

### Exercise — Sampling, Measure, and the Monte Carlo Integral

The current `compute_willmore` draws points **uniformly** from $[0, 2\pi]^2$.
This gives the estimator

$$W \approx \frac{(2\pi)^2}{N}\sum_{i=1}^{N} H_i^2\,\sqrt{E_iG_i - F_i^2},$$

because the domain area is $(2\pi)^2$ and the sample mean approximates
$\tfrac{1}{(2\pi)^2}\iint (\cdots)\,du\,dv$.

**What if you use a different sampling distribution $p(u,v)$?**

When samples are drawn from a non-uniform density $p$, the standard estimator is **wrong**.
By the Monte Carlo change-of-variables identity,

$$\iint f(u,v)\,du\,dv = \int \frac{f(u,v)}{p(u,v)}\,p(u,v)\,du\,dv
  \approx \frac{1}{N}\sum_{i=1}^{N} \frac{f(u_i,v_i)}{p(u_i,v_i)},
  \quad (u_i,v_i)\sim p.$$

So you must **divide each term by $p(u_i,v_i)$** — this is called *importance sampling*.

**Tasks:**

1. **Biased sampler** — Write a sampler that concentrates points near $v = 0$
   (the outer equator of the starting torus), e.g. by drawing $v \sim \mathcal{N}(0, 0.5)$
   wrapped to $[0, 2\pi]$, and leaving $u$ uniform.
   Call `compute_willmore` with this biased sample.
   Does the energy estimate change? Why?

2. **Corrected estimator** — Modify `compute_willmore` (or write a new function
   `compute_willmore_IS`) to accept a weight vector `weights` of shape `(N,)` where
   `weights[i] = 1 / p(u_i, v_i)` (un-normalised), so it computes

   $$W \approx \frac{1}{N}\sum_{i=1}^N w_i \cdot H_i^2\,\sqrt{E_iG_i-F_i^2}.$$

   Check that your corrected estimator recovers the same value as the uniform one
   on the analytic torus.

3. **Surface-area measure** — On the analytic torus the area element is
   $\sqrt{EG-F^2} = r(R + r\cos v)$, which *varies* with $v$.
   Uniform sampling in parameter space therefore over-represents the outer equator
   (large $R + r\cos v$) relative to the inner one.
   What sampling distribution $p^*(u,v) \propto r(R + r\cos v)$ would make
   each sample equally representative of surface area?
   *Hint:* the marginal in $u$ is still uniform; only the $v$-marginal changes.
   Is this a better or worse estimator for $W$?


In [ ]:
### WRITE EXERCISE CODE HERE ###

## 8 — Regularity Loss

Minimising the Willmore energy alone can cause the parametrisation to **degenerate**:
the network may try to collapse parts of the torus (area element $\to 0$) while
keeping $W$ finite.  This is the parametric analogue of a mesh folding over itself.

We prevent this with a **regularity loss** that penalises vanishing area elements:

$$\mathcal{L}_R = \frac{1}{N}\sum_i \max\!\bigl(0,\;\delta - \sqrt{E_i G_i - F_i^2}\bigr)^2,$$

where $\delta > 0$ is a small threshold (we use $\delta = 0.05$).  This is zero
whenever the parametrisation is non-degenerate and fires only when some region of
the surface is being compressed.

A well-conditioned torus of revolution has area element $r(R + r\cos v)$, which for
$(R=4, r=1)$ ranges from $r(R-r) = 3$ to $r(R+r) = 5$ — comfortably above $\delta$.

In [ ]:
MIN_AREA = 0.05    # collapse threshold δ


def compute_regularity(model: nn.Module, uv_batch: torch.Tensor,
                       min_area: float = MIN_AREA) -> torch.Tensor:
    """
    Penalise vanishing area element  √(EG − F²) < min_area.

    L_R = mean( relu(min_area − √(EG−F²))² )
    """
    _, phi_u, phi_v, _ = _first_derivs(model, uv_batch)

    E      = (phi_u * phi_u).sum(1)
    F_coef = (phi_u * phi_v).sum(1)
    G      = (phi_v * phi_v).sum(1)
    det    = torch.clamp(E * G - F_coef ** 2, min=EPS)
    area_el = det.sqrt()

    return torch.nn.functional.relu(min_area - area_el).pow(2).mean()


# ── Evaluate on the initial surface ──────────────────────────────────────────
L_R = compute_regularity(model, uv_eval)
print(f"Regularity loss on initial surface  L_R = {L_R.item():.6f}")
print("(Should be ≈ 0 because the initial surface is non-degenerate.)")

# Illustrate: area element along the equator slice v=0
v_slice = torch.zeros(500, 2, dtype=DTYPE)
v_slice[:, 0] = torch.linspace(0, TWO_PI, 500)
v_slice.requires_grad_(True)

phi_u_vals = torch.stack([
    torch.autograd.grad(model(v_slice)[:, k].sum(), v_slice,
                        retain_graph=True, create_graph=False)[0][:, 0]
    for k in range(3)
], dim=1).detach()   # (500, 3)  — φ_u along the equator

# Show the distribution of area elements on the initial surface
_, phi_u_all, phi_v_all, _ = _first_derivs(model, uv_eval)
E_all   = (phi_u_all * phi_u_all).sum(1).detach()
F_all   = (phi_u_all * phi_v_all).sum(1).detach()
G_all   = (phi_v_all * phi_v_all).sum(1).detach()
area_el_all = (E_all * G_all - F_all**2).clamp(min=EPS).sqrt()

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(area_el_all.numpy(), bins=50, color="steelblue", edgecolor="white")
ax.axvline(MIN_AREA, color="crimson", ls="--", label=f"threshold δ = {MIN_AREA}")
ax.set_xlabel("$\\sqrt{EG - F^2}$  (area element)")
ax.set_ylabel("count")
ax.set_title("Distribution of area elements — initial surface")
ax.legend(); plt.tight_layout(); plt.show()


### Exercise — Alternative Regularity Losses and the Embeddedness Problem

The current loss $\mathcal{L}_R = \mathrm{mean}(\mathrm{relu}(\delta - \sqrt{EG-F^2})^2)$
only prevents the area element from collapsing to zero.  It says nothing about the
*shape* of the parametrisation or whether the surface crosses itself.

---

#### 1. Orientation

The area element $\sqrt{EG-F^2} = |\varphi_u \times \varphi_v|$ is always non-negative,
so it cannot detect an **orientation reversal** (a local fold where the surface
tangent frame flips).  Numerically, the signed Jacobian determinant is

$$J = \varphi_u \times \varphi_v \cdot \hat{n}_0,$$

where $\hat{n}_0$ is some fixed reference normal.  When $J < 0$ the parametrisation
has locally flipped orientation — a precursor to self-intersection.

**Task A:** Write an orientation-preserving regularity term

$$\mathcal{L}_{\mathrm{orient}} = \mathrm{mean}\bigl(\mathrm{relu}(-J)^2\bigr)$$

that penalises any region where the signed Jacobian goes negative.
(*Hint:* compute `n_ref = torus_embed` normal at the same points and take the dot
product with $\varphi_u \times \varphi_v$, or simply use `area_el` — why is this
sufficient for detecting flips if the network stays close to the initial surface?)

---

#### 2. Other regularity losses

Beyond collapse and orientation, one may want to control the *quality* of the
parametrisation.  Some options:

- **Conformal energy** — penalise anisotropic stretching by adding
  $\mathcal{L}_C = \mathrm{mean}((E - G)^2 + 4F^2)$, which is zero iff
  $E = G$ and $F = 0$ (isothermal coordinates).  A conformal map preserves
  angles and spreads points more uniformly over the surface.

- **Isometric energy** — penalise $\mathcal{L}_I = \mathrm{mean}((E-1)^2 + 2F^2 + (G-1)^2)$
  to keep the parametrisation close to arc-length parametrised.  Combined with the
  Willmore loss this gives a well-conditioned problem, but may overconstrain the
  optimisation.

- **Dirichlet / gradient bound** — penalise $\mathrm{mean}(E + G)$ (total stretch)
  to prevent the network concentrating all resolution in a small region.

**Task B:** Implement $\mathcal{L}_C$ or $\mathcal{L}_I$ and evaluate it on the
pretrained surface.  Is it close to zero (as expected for the analytic torus in
standard parametrisation)?  What value do you expect analytically for $R=4$, $r=1$?

---

#### 3. The difficulty of preventing non-local self-intersections

All of the losses above are *local* — they are computed pointwise and can only detect
problems at individual sample locations.  They say nothing about whether two *distant*
parts of the surface collide:

$$\varphi(u_1, v_1) = \varphi(u_2, v_2), \quad (u_1,v_1) \neq (u_2,v_2).$$

Detecting this naively requires checking all $O(N^2)$ pairs of sample points — far
too expensive to include in a training loop.

**Discussion questions:**

- Why does a surface that is a local *immersion* everywhere (det $> 0$) still risk
  being a non-embedded surface (i.e., having self-intersections)?
  Think of a simple example.

- In our PINN, what prevents self-intersection in practice?
  (*Hint:* the supervised pretraining initialises the network to a known embedded
  surface; the Willmore energy itself diverges as a surface approaches a
  self-intersection — why?)

- Can you think of a tractable *approximate* penalty for self-intersection?
  For example: draw two independent batches $(u_i, v_i)$ and $(u'_j, v'_j)$,
  and penalise pairs where $\|\varphi(u_i,v_i) - \varphi(u'_j,v'_j)\|$ is small
  but $\|(u_i,v_i)-(u'_j,v'_j)\|$ is large.
  What are the computational and statistical difficulties with this approach?


In [ ]:
### WRITE EXERCISE CODE HERE ###

## 9 — PINN Training

We now minimise the **combined loss**

$$\mathcal{L} = \lambda_W\,\mathcal{L}_W + \lambda_R\,\mathcal{L}_R$$

with $\lambda_W = 1.0$ and $\lambda_R = 5.0$.

Key choices:

| Hyperparameter | Value | Reason |
|---------------|-------|--------|
| Optimiser | Adam | Works well for loss landscapes with scale separation |
| Learning rate | $3 \times 10^{-5}$ | Small enough to follow the energy surface smoothly |
| Batch size | 2 000 | Balance between gradient noise and per-step cost |
| Epochs | 500 | Enough to see a clear descent; ~2–3 min on CPU |
| Resample each epoch | yes | Resampling points avoid overfitting to a fixed grid |

Surface snapshots are saved every 100 epochs so we can watch the shape evolve.

In [ ]:
# ── Training hyperparameters ──────────────────────────────────────────────────
PINN_EPOCHS    = 500
PINN_LR        = 3e-5
PINN_BATCH     = 2000
LAMBDA_W       = 1.0    # Willmore weight
LAMBDA_R       = 5.0    # regularity weight
SNAPSHOT_EVERY = 100    # save surface snapshot every N epochs

# ── Grid used for surface snapshots ──────────────────────────────────────────
N_GRID = 50
u_grid = np.linspace(0, TWO_PI, N_GRID)
v_grid = np.linspace(0, TWO_PI, N_GRID)
Ug, Vg = np.meshgrid(u_grid, v_grid)
uv_grid = torch.tensor(
    np.stack([Ug.ravel(), Vg.ravel()], axis=1), dtype=DTYPE, device=DEVICE
)

def surface_snapshot(model: nn.Module) -> np.ndarray:
    """Return (N_GRID², 3) numpy array of the current surface on the grid."""
    with torch.no_grad():
        return model(uv_grid).numpy()

# ── Training ──────────────────────────────────────────────────────────────────
optimizer = optim.Adam(model.parameters(), lr=PINN_LR)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PINN_EPOCHS, eta_min=1e-7)

history_W, history_R = [], []
snapshots = {}   # epoch → (N_GRID², 3)

# Save initial snapshot
snapshots[0] = surface_snapshot(model)

print(f"{'Epoch':>6}  {'W (Willmore)':>14}  {'L_R (regularity)':>18}  {'LR':>10}")
print("-" * 56)

# Run training loop
for epoch in range(1, PINN_EPOCHS + 1):
    # Sample a new batch of points from the domain
    uv_batch = sample_torus_domain(PINN_BATCH)

    # Compute losses
    L_W = compute_willmore(model, uv_batch)
    L_R = compute_regularity(model, uv_batch)
    loss = LAMBDA_W * L_W + LAMBDA_R * L_R

    # Backprop and step
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()

    history_W.append(L_W.item())
    history_R.append(L_R.item())

    if epoch % 50 == 0:
        lr_now = scheduler.get_last_lr()[0]
        print(f"{epoch:6d}  {L_W.item():14.4f}  {L_R.item():18.6f}  {lr_now:.2e}")

    if epoch % SNAPSHOT_EVERY == 0:
        snapshots[epoch] = surface_snapshot(model)

print("\nTraining complete.")
print(f"Final Willmore estimate : {history_W[-1]:.4f}")
print(f"Target  (2π²)           : {2 * PI**2:.4f}")


### Exercise — Interpreting Results and Tuning the PINN

#### What is a "good" loss value?

The Willmore energy is a geometric quantity with a concrete lower bound:

$$W \geq 2\pi^2 \approx 19.74 \quad \text{for any embedded torus.}$$

This gives us an absolute reference — unlike most ML tasks where "good" is relative.

**Questions to think about:**

1. **Gap to the minimum** — After 500 epochs, compute the gap $W - 2\pi^2$.
   Is a gap of 5% "good"?  What about 1%?  At what point do you think numerical
   noise in the Monte Carlo estimate dominates over the true gap?

2. **Regularity loss** — The regularity loss $\mathcal{L}_R$ should stay near zero
   throughout training.  If it spikes, what does that indicate geometrically?
   What would happen to the surface if you set $\lambda_R = 0$?

#### Hyperparameter tuning

Re-run the training cell with different settings and record the final $W$ each time.
Some directions to explore:

| What to change | Suggestion | Expected effect |
|----------------|-----------|----------------|
| `PINN_EPOCHS` | 1000, 2000 | More descent at the cost of time |
| `PINN_LR` | `1e-4`, `1e-5` | Slower but more stable |
| `PINN_BATCH` | 500, 5000 | Noisier vs. more accurate gradients |
| `LAMBDA_W` / `LAMBDA_R` | try `LAMBDA_R = 1.0` | Less regularisation — does $W$ go lower? |

*Hint:* Re-run from the `torch.manual_seed(0)` cell (section 6) to reset the model
to the supervised pretrained state before each experiment, so comparisons start
from the same point.


In [ ]:
### WRITE EXERCISE CODE HERE ###

## 10 — Loss Curves and Surface Evolution

We now visualise:

1. **Loss curves** — Willmore energy and regularity loss vs. epoch.
2. **Surface snapshots** — the network's output surface at epochs 0, 100, 200, 300, 400, 500,
   showing how the shape deforms toward the Willmore minimiser.
3. **Final energy** — compare the PINN result to the theoretical value $W = 2\pi^2$.

In [ ]:
# ── Loss curves ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

epochs_ax = np.arange(1, PINN_EPOCHS + 1)

ax = axes[0]
ax.plot(epochs_ax, history_W, color="steelblue", lw=1.5, label="Willmore $W$")
ax.axhline(2 * PI**2, color="crimson", ls="--", lw=1.5, label=f"Target $2\\pi^2 = {2*PI**2:.2f}$")
ax.set_xlabel("Epoch"); ax.set_ylabel("$W$")
ax.set_title("Willmore energy vs. epoch")
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(epochs_ax, history_R, color="darkorange", lw=1.5)
ax.set_xlabel("Epoch"); ax.set_ylabel("$\\mathcal{L}_R$")
ax.set_title("Regularity loss vs. epoch")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# ── Surface evolution snapshots ───────────────────────────────────────────────
snap_epochs = sorted(snapshots.keys())
n_snaps = len(snap_epochs)

# Compute a shared axis range from all snapshot points
all_pts = np.concatenate([snapshots[ep] for ep in snap_epochs], axis=0)
x_all, y_all, z_all = all_pts[:, 0], all_pts[:, 1], all_pts[:, 2]
half = max(x_all.max() - x_all.min(),
           y_all.max() - y_all.min(),
           z_all.max() - z_all.min()) / 2
cx = (x_all.max() + x_all.min()) / 2
cy = (y_all.max() + y_all.min()) / 2
cz = (z_all.max() + z_all.min()) / 2

fig = plt.figure(figsize=(4 * n_snaps, 4))
for i, ep in enumerate(snap_epochs):
    xyz = snapshots[ep]           # (N_GRID², 3) — stored grid points
    X, Y, Z = xyz[:, 0], xyz[:, 1], xyz[:, 2]
    Z_norm = (Z - z_all.min()) / (z_all.max() - z_all.min() + 1e-10)

    ax = fig.add_subplot(1, n_snaps, i + 1, projection="3d")
    ax.scatter(X, Y, Z, c=Z_norm, cmap="coolwarm", s=2, alpha=0.6)
    W_ep = history_W[ep - 1] if ep > 0 else history_W[0]
    ax.set_title(f"Epoch {ep}\n$W={W_ep:.2f}$" if ep > 0 else
                 f"Epoch 0\n(pretrained)", fontsize=9)
    ax.set_xlabel("x", fontsize=7); ax.set_ylabel("y", fontsize=7)
    ax.set_zlabel("z", fontsize=7)
    ax.set_xlim(cx - half, cx + half)
    ax.set_ylim(cy - half, cy + half)
    ax.set_zlim(cz - half, cz + half)
    ax.set_box_aspect([1, 1, 1])
    ax.tick_params(labelsize=6)

plt.suptitle("Surface evolution during PINN training", y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

# ── Final comparison ──────────────────────────────────────────────────────────
W_final_mc = compute_willmore(model, sample_torus_domain(8000)).item()
W_target   = 2 * PI**2

print("─" * 50)
print(f"Initial Willmore  (R=4, r=1)  :  16π²/√15 ≈ {16*PI**2/np.sqrt(15):.4f}")
print(f"Final PINN estimate           :           {W_final_mc:.4f}")
print(f"Theoretical minimum (Clifford):  2π²     ≈ {W_target:.4f}")
print(f"Gap to minimum                :  {W_final_mc - W_target:.4f}  ({100*(W_final_mc/W_target-1):.1f} %)")


## Conclusion

In this tutorial you have used AI to numerically model a famous problem in Differential Geometry.

- You have chosen a geometric object to model with a neural network: the **embedding map**
- Your problem has a natural functional to minimise which you modelled with a loss: the **Willmore energy**
- This connects the AI learning process to a traditional geometric flow, such that the flow PDE was numerically approximated with this neural architecture's training; hence **"Neural PDEs"**

**What other problems could this methodology apply to?**
